In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# PSX40 Phase 2 — Kaggle CSV Import Test\n",
    "This notebook tests the full Phase 2 data import pipeline:\n",
    "1. Load CSV from `data/raw/kaggle/`\n",
    "2. Validate the data\n",
    "3. Insert into SQLite\n",
    "4. Show `db_summary()`\n",
    "5. Run `ScreenerEngine` on real data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "\n",
    "# Make sure project root is on the path\n",
    "sys.path.insert(0, os.path.abspath('..'))\n",
    "\n",
    "from pathlib import Path\n",
    "import pandas as pd"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1 — Check what CSV files are in data/raw/kaggle/"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.ingestion.kaggle_csv_loader import KAGGLE_DIR\n",
    "\n",
    "KAGGLE_DIR.mkdir(parents=True, exist_ok=True)\n",
    "csv_files = sorted(KAGGLE_DIR.glob('*.csv'))\n",
    "\n",
    "if not csv_files:\n",
    "    print(f'No CSV files found in {KAGGLE_DIR}')\n",
    "    print('Place your Kaggle CSV file(s) in data/raw/kaggle/ and re-run.')\n",
    "else:\n",
    "    for f in csv_files:\n",
    "        size_kb = f.stat().st_size / 1024\n",
    "        print(f'  {f.name:40s}  ({size_kb:.1f} KB)')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2 — Load CSV (auto-detects combined vs per-ticker)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.ingestion.kaggle_csv_loader import load_all_from_kaggle_dir\n",
    "\n",
    "raw_df = load_all_from_kaggle_dir()\n",
    "\n",
    "if raw_df.empty:\n",
    "    print('No data loaded. Using sample data fallback.')\n",
    "else:\n",
    "    print(f'\\nLoaded shape: {raw_df.shape}')\n",
    "    print(f'Tickers found: {sorted(raw_df[\"ticker\"].unique())}')\n",
    "    display(raw_df.head(10))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3 — Validate"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.ingestion.data_validator import validate_ohlcv\n",
    "\n",
    "if not raw_df.empty:\n",
    "    clean_df, result = validate_ohlcv(raw_df, source='kaggle_csv', auto_clean=True)\n",
    "    print(result.summary())\n",
    "    print(f'\\nClean rows ready to insert: {len(clean_df)}')\n",
    "    display(clean_df.head())\n",
    "else:\n",
    "    clean_df = pd.DataFrame()\n",
    "    print('Skipped — no data to validate.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4 — Insert into SQLite"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.database.connection import init_db\n",
    "from src.database.db_manager import upsert_prices, db_summary\n",
    "\n",
    "init_db()\n",
    "\n",
    "if not clean_df.empty:\n",
    "    upsert_prices(clean_df)\n",
    "    print('Insert complete.')\n",
    "else:\n",
    "    print('Nothing to insert.')\n",
    "\n",
    "summary = db_summary()\n",
    "print('\\nDatabase summary:')\n",
    "for k, v in summary.items():\n",
    "    print(f'  {k}: {v}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5 — Run ScreenerEngine on imported data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.screener.engine import ScreenerEngine\n",
    "\n",
    "engine = ScreenerEngine()\n",
    "engine.run(force_reseed=False)   # False = use existing DB data, no re-seed\n",
    "\n",
    "screener_df = engine.get_screener_df()\n",
    "print(f'\\nScored {len(screener_df)} stocks')\n",
    "display(screener_df[[\n",
    "    'ticker', 'sector', 'latest_close',\n",
    "    'sma_20', 'sma_50', 'rsi_14',\n",
    "    'volume_ratio', 'demo_score', 'verdict'\n",
    "]].head(15))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6 — Quick price chart for one ticker"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "\n",
    "tickers = engine.get_tickers()\n",
    "ticker  = tickers[0] if tickers else None\n",
    "\n",
    "if ticker:\n",
    "    price_df = engine.get_price_df(ticker)\n",
    "\n",
    "    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,\n",
    "                        row_heights=[0.65, 0.35], vertical_spacing=0.07)\n",
    "\n",
    "    fig.add_trace(go.Scatter(x=price_df['date'], y=price_df['close'],\n",
    "                             name='Close', line=dict(color='#2563eb')), row=1, col=1)\n",
    "    fig.add_trace(go.Scatter(x=price_df['date'], y=price_df['sma_20'],\n",
    "                             name='SMA 20', line=dict(color='#f59e0b')), row=1, col=1)\n",
    "    fig.add_trace(go.Scatter(x=price_df['date'], y=price_df['sma_50'],\n",
    "                             name='SMA 50', line=dict(color='#7c3aed', dash='dot')), row=1, col=1)\n",
    "    fig.add_trace(go.Scatter(x=price_df['date'], y=price_df['rsi_14'],\n",
    "                             name='RSI 14', line=dict(color='#0891b2')), row=2, col=1)\n",
    "    fig.add_hline(y=70, line_dash='dash', line_color='red',   row=2, col=1)\n",
    "    fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)\n",
    "\n",
    "    fig.update_layout(title=f'{ticker} — Phase 2 Import', height=500)\n",
    "    fig.show()\n",
    "else:\n",
    "    print('No tickers available to chart.')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'null' is not defined